# U5.1 — Preparación de PySpark y exploración de NYC Taxi

**Materia:** Tópicos de Big Data  
**Objetivo:** comprobar PySpark en VS Code y explorar una muestra real de NYC Yellow Taxi sin guardar el dataset completo en la computadora.

**Alcance:** instalación, SparkSession, DataFrames y lectura remota de Parquet. Sin Databricks, limpieza ni modelos de ML.

**Fuente:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), Yellow Taxi, enero de 2025.

**Nota:** Spark se ejecuta localmente. PyArrow leerá por HTTP los fragmentos del Parquet necesarios; esto **no equivale a procesamiento distribuido remoto** y sí consume red y memoria. No se crea un archivo de datos local.


## 1. Preparar y comprobar el entorno

Selecciona el kernel Python 3.12 de `.venv` en VS Code. Si faltan paquetes, instala en la terminal:

```bash
python -m pip install pyspark ipykernel pyarrow
```

Comprueba Python, PySpark, PyArrow y Java ejecutando la siguiente celda. 


In [1]:
import sys
import subprocess
import pyspark
import pyarrow

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("PyArrow:", pyarrow.__version__)
java = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java:", (java.stderr or java.stdout).splitlines()[0])


Python: 3.12.0
PySpark: 4.3.0
PyArrow: 26.0.1
Java: java version "26.0.2.1" 2026-08-18


## 2. Crear una SparkSession

`local[*]` utiliza los núcleos disponibles de esta computadora.


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("U5_1_NYC_Taxi")
    .master("local[*]")
    .getOrCreate()
)

# La instrucción local[*] indica que Spark debe ejecutarse localmente, utilizando los núcleos disponibles de tu computadora.
# En Databricks no necesitaremos especificar master("local[*]"), porque la plataforma administra el entorno de ejecución.

print("Versión de Spark:", spark.version)
print("SparkSession creada correctamente")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/09 13:50:33 WARN Utils: Your hostname, MacBook-Pro-de-Lizette.local, resolves to a loopback address: 127.0.0.1; using 172.22.18.156 instead (on interface en0)
26/11/09 13:50:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 13:50:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/09 13:50:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Versión de Spark: 4.3.0
SparkSession creada correctamente


## 3. Probar un DataFrame pequeño

Comprobamos que Spark puede procesar registros y mostrar su esquema.


In [4]:
datos = [
    (1, 3.5, 15.0),
    (2, 8.2, 32.5),
    (3, 1.8, 10.0)
]

columnas = ["viaje_id", "distancia", "tarifa"]

df = spark.createDataFrame(datos, columnas)

df.show()
df.printSchema()

+--------+---------+------+
|viaje_id|distancia|tarifa|
+--------+---------+------+
|       1|      3.5|  15.0|
|       2|      8.2|  32.5|
|       3|      1.8|  10.0|
+--------+---------+------+

root
 |-- viaje_id: long (nullable = true)
 |-- distancia: double (nullable = true)
 |-- tarifa: double (nullable = true)



## 4. Conocer la fuente remota

El archivo original está publicado en **Parquet**, un formato columnar. Consultaremos su estructura mediante **PyArrow**, leyendo rangos HTTP del archivo sin guardarlo en disco.

**¿Por qué no `spark.read.parquet(URL)`?** Spark local no interpreta directamente una URL HTTPS común como un sistema de archivos Parquet. Para esta práctica introductoria, PyArrow leerá una parte del archivo y luego Spark procesará una muestra pequeña.

El servidor debe admitir solicitudes HTTP `Range`. Si no las admite o falla la conexión, aparecerá un error claro.


In [5]:
from urllib.request import Request, urlopen
import io
import pyarrow.parquet as pq

URL_NYC = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

class ArchivoHTTPPorRangos(io.RawIOBase):
    """Archivo de solo lectura que solicita únicamente rangos de bytes por HTTP."""
    def __init__(self, url):
        self.url = url
        with urlopen(Request(url, method="HEAD"), timeout=30) as r:
            self.tamano = int(r.headers["Content-Length"])
        self.posicion = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.posicion

    def seek(self, desplazamiento, origen=io.SEEK_SET):
        if origen == io.SEEK_SET:
            nueva = desplazamiento
        elif origen == io.SEEK_CUR:
            nueva = self.posicion + desplazamiento
        elif origen == io.SEEK_END:
            nueva = self.tamano + desplazamiento
        else:
            raise ValueError("Origen de búsqueda no válido")
        if nueva < 0:
            raise ValueError("Posición negativa")
        self.posicion = nueva
        return self.posicion

    def readinto(self, buffer):
        if self.posicion >= self.tamano:
            return 0
        cantidad = min(len(buffer), self.tamano - self.posicion)
        fin = self.posicion + cantidad - 1
        solicitud = Request(self.url, headers={"Range": f"bytes={self.posicion}-{fin}"})
        with urlopen(solicitud, timeout=60) as respuesta:
            if respuesta.status != 206:
                raise RuntimeError("El servidor no admite lectura HTTP por rangos (206).")
            datos = respuesta.read(cantidad)
        buffer[:len(datos)] = datos
        self.posicion += len(datos)
        return len(datos)

archivo_remoto = io.BufferedReader(ArchivoHTTPPorRangos(URL_NYC), buffer_size=1024 * 1024)
parquet = pq.ParquetFile(archivo_remoto)

print("Grupos de filas:", parquet.num_row_groups)
print("Filas totales declaradas:", parquet.metadata.num_rows)
print("Columnas:", parquet.schema.names)


Grupos de filas: 4
Filas totales declaradas: 3475226
Columnas: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


## 5. Leer una muestra sin descargar el dataset completo

Leeremos **solo tres columnas del primer grupo de filas** y tomaremos sus primeros 1,000 registros. Un grupo de filas puede contener muchos más registros: PyArrow debe transferir las partes necesarias de esas columnas, por lo que **la transferencia puede ser mayor que 1,000 filas**.

El resultado sí pasa a la memoria local para crear el DataFrame de PySpark.


In [6]:
columnas_nyc = ["VendorID", "trip_distance", "total_amount"]

# PyArrow lee las columnas seleccionadas del primer grupo de filas.
tabla_arrow = parquet.read_row_group(0, columns=columnas_nyc)
muestra = tabla_arrow.slice(0, 1000)

# Convertimos la muestra a registros Python para crear un DataFrame Spark.
df_taxi = spark.createDataFrame(muestra.to_pylist())

print("Registros de la muestra:", df_taxi.count())
df_taxi.printSchema()
df_taxi.show(5)


Registros de la muestra: 1000
root
 |-- VendorID: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)

+--------+------------+-------------+
|VendorID|total_amount|trip_distance|
+--------+------------+-------------+
|       1|        18.0|          1.6|
|       1|       12.12|          0.5|
|       1|        12.1|          0.6|
|       2|         9.7|         0.52|
|       2|         8.3|         0.66|
+--------+------------+-------------+
only showing top 5 rows


## 6. Primera exploración con PySpark

Utilizamos operaciones básicas: selección, filtro y agregación. **Los resultados corresponden a la muestra, no a todos los viajes de enero.**


In [7]:
from pyspark.sql import functions as F

# Mostrar viajes con distancia positiva.
df_taxi.filter(F.col("trip_distance") > 0).show(5)

# Resumen sencillo de la muestra.
df_taxi.select(
    F.count("*").alias("viajes_muestra"),
    F.round(F.avg("trip_distance"), 2).alias("distancia_promedio"),
    F.round(F.avg("total_amount"), 2).alias("importe_promedio")
).show()


+--------+------------+-------------+
|VendorID|total_amount|trip_distance|
+--------+------------+-------------+
|       1|        18.0|          1.6|
|       1|       12.12|          0.5|
|       1|        12.1|          0.6|
|       2|         9.7|         0.52|
|       2|         8.3|         0.66|
+--------+------------+-------------+
only showing top 5 rows
+--------------+------------------+----------------+
|viajes_muestra|distancia_promedio|importe_promedio|
+--------------+------------------+----------------+
|          1000|              2.78|           24.93|
+--------------+------------------+----------------+



## 7. Comprobación y reflexión

1. ¿Qué hace `SparkSession`?
2. ¿Cuál es la diferencia entre `df.show()` y `df.printSchema()`?
3. ¿Qué biblioteca accedió al archivo Parquet remoto y cuál procesó la muestra?
4. ¿Por qué no podemos afirmar que procesamos todo el dataset de NYC Taxi?
5. ¿En qué computadora se ejecutaron las operaciones de Spark?

**Resultado esperado:** una sesión de Spark funcional, metadatos del Parquet remoto y un DataFrame de muestra procesado localmente, sin guardar el dataset en disco.
